In [ ]:
# نصب کتابخانه‌های لازم

import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "catboost": "catboost",
    "joblib": "joblib",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {package_name} ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

print("All required packages are available.")

In [ ]:
from pathlib import Path
from itertools import combinations
import gc
import hashlib
import json
import math
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 1. تنظیمات اجرا

In [ ]:
RUN_MODE = "pilot"  # "pilot" or "full"

# Pilot
PILOT_OUTER_FOLDS = [1]
PILOT_MODEL_FAMILIES = [
    "RandomForest",
    "CatBoost",
]
PILOT_FEATURE_SETS = [
    "B1_PRIMARY_LOGS",
    "B2_DOMAIN_GRAPH",
]
PILOT_MAX_ROWS_PER_WELL = 2500
PILOT_RF_N_ESTIMATORS = 120
PILOT_CATBOOST_MAX_ITERATIONS = 180
PILOT_CATBOOST_EARLY_STOPPING = 30

# Full
FULL_OUTER_FOLDS = [1, 2, 3, 4, 5]

# Random Forest مدل اصلی روی تمام Feature setها
FULL_RF_FEATURE_SETS = [
    "B0_CONTEXT_ONLY",
    "B1_PRIMARY_LOGS",
    "B2_DOMAIN_GRAPH",
    "B3_ALL_LOGS_NO_SPATIAL",
    "B4_FULL_SPATIAL",
]

# CatBoost فقط به‌عنوان مقایسه اصلی روی سه طراحی کلیدی
FULL_CATBOOST_FEATURE_SETS = [
    "B1_PRIMARY_LOGS",
    "B2_DOMAIN_GRAPH",
    "B4_FULL_SPATIAL",
]

FULL_RF_N_ESTIMATORS = 800
FULL_CATBOOST_MAX_ITERATIONS = 700
FULL_CATBOOST_EARLY_STOPPING = 75


RF_MAX_DEPTH = 200
RF_MIN_SAMPLES_SPLIT = 5
RF_MIN_SAMPLES_LEAF = 2
RF_N_JOBS = -1
RF_MAX_FEATURES = "sqrt"
RF_BOOTSTRAP = True

# CatBoost comparator
CATBOOST_DEPTH = 8
CATBOOST_LEARNING_RATE = 0.05
CATBOOST_L2_LEAF_REG = 5.0
CATBOOST_RANDOM_STRENGTH = 1.0
CATBOOST_BORDER_COUNT = 128
CATBOOST_THREADS = -1
CATBOOST_VERBOSE_EVERY = 100
MIN_CATBOOST_REFIT_ITERATIONS = 40

SAVE_OOF_PROBABILITIES = True
SKIP_COMPLETED_RUNS = True

if RUN_MODE == "pilot":
    ACTIVE_OUTER_FOLDS = PILOT_OUTER_FOLDS
    ACTIVE_RUNS = [
        (model_family, feature_set)
        for model_family in PILOT_MODEL_FAMILIES
        for feature_set in PILOT_FEATURE_SETS
    ]
    RF_N_ESTIMATORS = PILOT_RF_N_ESTIMATORS
    CATBOOST_MAX_ITERATIONS = (
        PILOT_CATBOOST_MAX_ITERATIONS
    )
    CATBOOST_EARLY_STOPPING = (
        PILOT_CATBOOST_EARLY_STOPPING
    )
else:
    ACTIVE_OUTER_FOLDS = FULL_OUTER_FOLDS
    ACTIVE_RUNS = (
        [
            ("RandomForest", feature_set)
            for feature_set in FULL_RF_FEATURE_SETS
        ]
        +
        [
            ("CatBoost", feature_set)
            for feature_set in FULL_CATBOOST_FEATURE_SETS
        ]
    )
    RF_N_ESTIMATORS = FULL_RF_N_ESTIMATORS
    CATBOOST_MAX_ITERATIONS = (
        FULL_CATBOOST_MAX_ITERATIONS
    )
    CATBOOST_EARLY_STOPPING = (
        FULL_CATBOOST_EARLY_STOPPING
    )

print("RUN_MODE:", RUN_MODE)
print("Outer folds:", ACTIVE_OUTER_FOLDS)
print("Runs:")
for run in ACTIVE_RUNS:
    print("  ", run)

print("\nRF trees:", RF_N_ESTIMATORS)
print("CatBoost max iterations:", CATBOOST_MAX_ITERATIONS)

In [ ]:
# Portable configuration (repository root or notebooks/ working directory)
REPO_ROOT = Path(os.environ.get("FORCE_REPO_ROOT", Path.cwd())).expanduser().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "notebooks").is_dir():
    raise FileNotFoundError(
        "Repository root not found. Open Jupyter from the repository root "
        "or notebooks/, or set FORCE_REPO_ROOT to the repo path."
    )
DATA_DIR = Path(os.environ.get("FORCE_DATA_DIR", str(REPO_ROOT / "data" / "raw"))).expanduser().resolve()
PREPARED_DIR = Path(os.environ.get("FORCE_PREPARED_DIR", str(REPO_ROOT / "data" / "prepared"))).expanduser().resolve()
RESULTS_ROOT = Path(os.environ.get("FORCE_RESULTS_DIR", str(REPO_ROOT / "outputs"))).expanduser().resolve()

PROJECT_ROOT = RESULTS_ROOT
STAGE_01_DIR = PREPARED_DIR
STAGE_01B_DIR = PREPARED_DIR
CLEAN_DATA_PATH = STAGE_01_DIR / "FORCE_train_cleaned_with_splits.parquet"
CONTRACT_PATH = STAGE_01B_DIR / "07_causal_project_contract_v0_2_1.json"
OUTPUT_DIR = PROJECT_ROOT / "02_baseline_RF_primary_CatBoost_comparator" / RUN_MODE
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "oof_predictions"
METRIC_DIR = OUTPUT_DIR / "fold_metrics"
IMPORTANCE_DIR = OUTPUT_DIR / "feature_importance"
CONFIG_DIR = OUTPUT_DIR / "run_metadata"
RESAMPLING_DIR = OUTPUT_DIR / "resampling_reports"

for directory in [
    OUTPUT_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    METRIC_DIR,
    IMPORTANCE_DIR,
    CONFIG_DIR,
    RESAMPLING_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(CLEAN_DATA_PATH)

if not CONTRACT_PATH.exists():
    raise FileNotFoundError(CONTRACT_PATH)

with open(
    CONTRACT_PATH,
    "r",
    encoding="utf-8",
) as file:
    contract = json.load(file)

if not contract.get("baseline_ready", False):
    raise RuntimeError(
        "The causal contract is not baseline-ready."
    )

print("Clean data:", CLEAN_DATA_PATH)
print("Contract  :", CONTRACT_PATH)
print("Output    :", OUTPUT_DIR)
print("Contract version:", contract["contract_version"])

In [ ]:
data = pd.read_parquet(
    CLEAN_DATA_PATH
)

required_columns = {
    "ROW_ID",
    "WELL",
    "DEPTH_MD",
    "target",
    "GROUP_FOLD",
    "GEO_BLOCK",
}

missing_required = sorted(
    required_columns
    - set(data.columns)
)

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

data = (
    data
    .sort_values("ROW_ID")
    .reset_index(drop=True)
)

print("Rows   :", len(data))
print("Wells  :", data["WELL"].nunique())
print("Classes:", data["target"].nunique())
print("Folds  :", sorted(data["GROUP_FOLD"].unique()))

In [ ]:
LITHOLOGY_NAME_MAP = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

CLASS_CODES = sorted(
    int(value)
    for value in data["target"].unique()
)

CLASS_TO_INDEX = {
    class_code: index
    for index, class_code
    in enumerate(CLASS_CODES)
}

INDEX_TO_CLASS = {
    index: class_code
    for class_code, index
    in CLASS_TO_INDEX.items()
}

CLASS_NAMES = [
    LITHOLOGY_NAME_MAP[class_code]
    for class_code in CLASS_CODES
]

data["target_index"] = (
    data["target"]
    .map(CLASS_TO_INDEX)
    .astype("int16")
)

class_mapping = pd.DataFrame({
    "class_index": list(range(len(CLASS_CODES))),
    "lithology_code": CLASS_CODES,
    "lithology_name": CLASS_NAMES,
})

display(class_mapping)

In [ ]:
PRIMARY_DIRECT_FEATURES = [
    feature
    for feature in contract[
        "primary_direct_treatment_features"
    ]
    if feature in data.columns
]

PATHWAY_FEATURES = [
    feature
    for feature in contract[
        "pathway_driver_features"
    ]
    if feature in data.columns
]

ALL_PHYSICAL_LOGS = [
    feature
    for feature in contract[
        "feature_roles"
    ]["physical_logs"]
    if feature in data.columns
]

SPATIAL_COLUMNS = [
    feature
    for feature in contract[
        "feature_roles"
    ]["spatial_context"]
    if feature in data.columns
]

STRATIGRAPHIC_COLUMNS = [
    feature
    for feature in contract[
        "feature_roles"
    ]["stratigraphic_context"]
    if feature in data.columns
]

PRIOR_PAPER_BASE_FEATURES = [
    feature
    for feature in contract[
        "prior_paper_base_features"
    ]
    if feature in data.columns
]

FEATURE_SET_DEFINITIONS = {
    "B0_CONTEXT_ONLY": {
        "raw_numeric": [
            feature
            for feature in [
                "DEPTH_MD",
                *SPATIAL_COLUMNS,
            ]
            if feature in data.columns
        ],
        "categorical": STRATIGRAPHIC_COLUMNS,
        "engineered_prior_features": False,
        "add_missing_indicators": True,
        "scientific_role": (
            "context-only shortcut diagnostic"
        ),
    },
    "B1_PRIMARY_LOGS": {
        "raw_numeric": PRIMARY_DIRECT_FEATURES,
        "categorical": [],
        "engineered_prior_features": False,
        "add_missing_indicators": True,
        "scientific_role": (
            "primary measurement baseline"
        ),
    },
    "B2_DOMAIN_GRAPH": {
        "raw_numeric": list(dict.fromkeys(
            PRIMARY_DIRECT_FEATURES
            + PATHWAY_FEATURES
        )),
        "categorical": [],
        "engineered_prior_features": True,
        "add_missing_indicators": True,
        "scientific_role": (
            "prior-paper domain-guided feature design"
        ),
    },
    "B3_ALL_LOGS_NO_SPATIAL": {
        "raw_numeric": list(dict.fromkeys(
            ALL_PHYSICAL_LOGS
            + ["DEPTH_MD"]
        )),
        "categorical": STRATIGRAPHIC_COLUMNS,
        "engineered_prior_features": False,
        "add_missing_indicators": True,
        "scientific_role": (
            "all measurements without XYZ coordinates"
        ),
    },
    "B4_FULL_SPATIAL": {
        "raw_numeric": list(dict.fromkeys(
            ALL_PHYSICAL_LOGS
            + ["DEPTH_MD"]
            + SPATIAL_COLUMNS
        )),
        "categorical": STRATIGRAPHIC_COLUMNS,
        "engineered_prior_features": False,
        "add_missing_indicators": True,
        "scientific_role": (
            "full-context spatial shortcut diagnostic"
        ),
    },
}

feature_set_table = pd.DataFrame([
    {
        "feature_set": name,
        "raw_numeric_count": len(
            definition["raw_numeric"]
        ),
        "categorical_count": len(
            definition["categorical"]
        ),
        "engineered_prior_features": (
            definition[
                "engineered_prior_features"
            ]
        ),
        "scientific_role": (
            definition["scientific_role"]
        ),
    }
    for name, definition
    in FEATURE_SET_DEFINITIONS.items()
])

display(feature_set_table)

In [ ]:
RATIO_DEFINITIONS = {
    "NPHI_RHOB_RATIO": (
        "NPHI",
        "RHOB",
    ),
    "GR_RHOB_RATIO": (
        "GR",
        "RHOB",
    ),
    "DTC_RHOB_RATIO": (
        "DTC",
        "RHOB",
    ),
}

def prepare_feature_set(
    source_dataframe,
    feature_set_name,
):
    definition = (
        FEATURE_SET_DEFINITIONS[
            feature_set_name
        ]
    )

    raw_numeric = [
        column
        for column in definition[
            "raw_numeric"
        ]
        if column in source_dataframe.columns
    ]

    categorical = [
        column
        for column in definition[
            "categorical"
        ]
        if column in source_dataframe.columns
    ]

    feature_frame = pd.DataFrame(
        index=source_dataframe.index
    )

    for column in raw_numeric:
        feature_frame[column] = (
            pd.to_numeric(
                source_dataframe[column],
                errors="coerce",
            )
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .astype("float32")
        )

    for column in categorical:
        feature_frame[column] = (
            source_dataframe[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    if definition[
        "engineered_prior_features"
    ]:
        available_prior = [
            feature
            for feature in PRIOR_PAPER_BASE_FEATURES
            if feature in feature_frame.columns
        ]

        for first, second in combinations(
            available_prior,
            2,
        ):
            feature_frame[
                f"{first}_X_{second}"
            ] = (
                feature_frame[first]
                * feature_frame[second]
            ).astype("float32")

        for feature in available_prior:
            feature_frame[
                f"{feature}_SQUARED"
            ] = (
                feature_frame[feature] ** 2
            ).astype("float32")

            feature_frame[
                f"{feature}_LOGABS"
            ] = (
                np.log1p(
                    np.abs(
                        feature_frame[feature]
                    )
                )
            ).astype("float32")

        epsilon = np.float32(1e-6)

        for new_name, (
            numerator,
            denominator,
        ) in RATIO_DEFINITIONS.items():
            if (
                numerator in feature_frame.columns
                and denominator in feature_frame.columns
            ):
                feature_frame[new_name] = (
                    feature_frame[numerator]
                    / (
                        np.abs(
                            feature_frame[denominator]
                        )
                        + epsilon
                    )
                ).astype("float32")

    if definition[
        "add_missing_indicators"
    ]:
        for column in raw_numeric:
            if column in feature_frame.columns:
                feature_frame[
                    f"{column}__MISSING"
                ] = (
                    feature_frame[column]
                    .isna()
                    .astype("int8")
                )

    categorical_names = [
        column
        for column in categorical
        if column in feature_frame.columns
    ]

    numeric_names = [
        column
        for column in feature_frame.columns
        if column not in categorical_names
    ]

    if feature_frame.columns.duplicated().any():
        duplicated = (
            feature_frame.columns[
                feature_frame.columns.duplicated()
            ]
            .tolist()
        )
        raise RuntimeError(
            f"Duplicated features: {duplicated}"
        )

    return (
        feature_frame,
        numeric_names,
        categorical_names,
    )


preview, preview_numeric, preview_categorical = (
    prepare_feature_set(
        data.head(1000),
        "B2_DOMAIN_GRAPH",
    )
)

print("B2 preview shape:", preview.shape)
print("Numeric:", len(preview_numeric))
print("Categorical:", preview_categorical)
print(preview.columns.tolist())

del preview
gc.collect()

In [ ]:
def deterministic_sample_per_well(
    dataframe,
    max_rows_per_well,
    random_state,
):
    sampled_parts = []

    for well_name, well_df in dataframe.groupby(
        "WELL",
        sort=True,
    ):
        if len(well_df) <= max_rows_per_well:
            sampled_parts.append(well_df)
            continue

        seed_text = (
            f"{random_state}|{well_name}"
        )

        seed_value = int(
            hashlib.sha256(
                seed_text.encode("utf-8")
            ).hexdigest()[:8],
            16,
        )

        sampled_parts.append(
            well_df.sample(
                n=max_rows_per_well,
                random_state=seed_value,
            )
        )

    return (
        pd.concat(
            sampled_parts,
            axis=0,
        )
        .sort_values("ROW_ID")
        .reset_index(drop=True)
    )


if RUN_MODE == "pilot":
    modeling_data = (
        deterministic_sample_per_well(
            data,
            max_rows_per_well=(
                PILOT_MAX_ROWS_PER_WELL
            ),
            random_state=RANDOM_STATE,
        )
    )
else:
    modeling_data = data

print("Modeling rows:", len(modeling_data))
print("Modeling wells:", modeling_data["WELL"].nunique())
display(
    modeling_data.groupby(
        "GROUP_FOLD"
    ).size()
)

In [ ]:
UNDERSAMPLE_FRACTIONS = {
    65000: 0.30,  # Shale
    30000: 0.20,  # Sandstone
    70000: 0.20,  # Limestone
}

OVERSAMPLE_MULTIPLIERS = {
    93000: 5.0,   # Basement
    74000: 1.5,   # Dolomite
}

def apply_prior_paper_hybrid_resampling(
    training_meta,
    random_state,
):
    rng = np.random.default_rng(
        random_state
    )

    selected_positions = []
    report_rows = []

    y_codes = (
        training_meta["target"]
        .to_numpy(dtype=int)
    )

    for class_code in CLASS_CODES:
        class_positions = np.flatnonzero(
            y_codes == class_code
        )

        original_count = len(
            class_positions
        )

        if original_count == 0:
            report_rows.append({
                "lithology_code": class_code,
                "lithology_name": (
                    LITHOLOGY_NAME_MAP[
                        class_code
                    ]
                ),
                "original_count": 0,
                "resampled_count": 0,
                "operation": "class_absent",
            })
            continue

        if class_code in UNDERSAMPLE_FRACTIONS:
            fraction = (
                UNDERSAMPLE_FRACTIONS[
                    class_code
                ]
            )

            target_count = max(
                1,
                int(
                    math.floor(
                        original_count
                        * fraction
                    )
                ),
            )

            chosen = rng.choice(
                class_positions,
                size=target_count,
                replace=False,
            )

            operation = (
                f"undersample_to_{fraction:.2f}"
            )

        elif class_code in OVERSAMPLE_MULTIPLIERS:
            multiplier = (
                OVERSAMPLE_MULTIPLIERS[
                    class_code
                ]
            )

            target_count = max(
                original_count,
                int(
                    math.ceil(
                        original_count
                        * multiplier
                    )
                ),
            )

            extra_count = (
                target_count
                - original_count
            )

            if extra_count > 0:
                extra_positions = rng.choice(
                    class_positions,
                    size=extra_count,
                    replace=True,
                )

                chosen = np.concatenate([
                    class_positions,
                    extra_positions,
                ])
            else:
                chosen = class_positions.copy()

            operation = (
                f"oversample_to_{multiplier:.2f}x"
            )

        else:
            chosen = class_positions.copy()
            target_count = original_count
            operation = "unchanged"

        selected_positions.append(
            chosen
        )

        report_rows.append({
            "lithology_code": class_code,
            "lithology_name": (
                LITHOLOGY_NAME_MAP[
                    class_code
                ]
            ),
            "original_count": int(
                original_count
            ),
            "resampled_count": int(
                len(chosen)
            ),
            "operation": operation,
        })

    if not selected_positions:
        raise RuntimeError(
            "No rows remained after resampling."
        )

    resampled_positions = np.concatenate(
        selected_positions
    )

    rng.shuffle(
        resampled_positions
    )

    report = pd.DataFrame(
        report_rows
    )

    return (
        resampled_positions,
        report,
    )

In [ ]:
def split_score(
    fold_class_counts,
    fold_row_counts,
    class_weights,
):
    class_totals = (
        fold_class_counts.sum(axis=0)
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore",
    ):
        class_fractions = np.divide(
            fold_class_counts,
            class_totals,
            out=np.zeros_like(
                fold_class_counts,
                dtype=float,
            ),
            where=class_totals > 0,
        )

    class_imbalance = float(
        np.average(
            np.std(
                class_fractions,
                axis=0,
            ),
            weights=class_weights,
        )
    )

    row_fractions = (
        fold_row_counts
        / max(
            fold_row_counts.sum(),
            1,
        )
    )

    return (
        class_imbalance
        + 0.25
        * float(
            np.std(row_fractions)
        )
        + 10.0
        * float(
            np.sum(
                fold_row_counts == 0
            )
        )
    )


def build_grouped_inner_folds(
    dataframe,
    n_folds=3,
    random_state=42,
    n_restarts=100,
):
    well_class_counts = pd.crosstab(
        dataframe["WELL"],
        dataframe["target_index"],
    ).sort_index()

    matrix = well_class_counts.to_numpy(
        dtype=float
    )

    wells = (
        well_class_counts.index
        .to_numpy()
    )

    n_wells, n_classes = matrix.shape

    class_well_frequency = (
        matrix > 0
    ).sum(axis=0)

    class_weights = (
        1.0
        / np.sqrt(
            np.maximum(
                class_well_frequency,
                1,
            )
        )
    )

    class_weights = (
        class_weights
        / class_weights.mean()
    )

    priority = (
        (
            (matrix > 0)
            * class_weights
        ).sum(axis=1)
        + (
            matrix.sum(axis=1)
            / max(
                matrix.sum(),
                1,
            )
        )
    )

    best_assignment = None
    best_score = np.inf

    for restart in range(n_restarts):
        rng = np.random.default_rng(
            random_state + restart
        )

        order = np.argsort(
            -(
                priority
                + rng.uniform(
                    0,
                    1e-6,
                    size=n_wells,
                )
            )
        )

        fold_class_counts = np.zeros(
            (n_folds, n_classes),
            dtype=float,
        )

        fold_row_counts = np.zeros(
            n_folds,
            dtype=float,
        )

        assignment = np.full(
            n_wells,
            -1,
            dtype=int,
        )

        for well_index in order:
            candidate_scores = []

            for fold_index in range(n_folds):
                temp_class = (
                    fold_class_counts.copy()
                )

                temp_rows = (
                    fold_row_counts.copy()
                )

                temp_class[
                    fold_index
                ] += matrix[well_index]

                temp_rows[
                    fold_index
                ] += matrix[
                    well_index
                ].sum()

                candidate_scores.append(
                    split_score(
                        temp_class,
                        temp_rows,
                        class_weights,
                    )
                )

            minimum = min(candidate_scores)

            tied = [
                fold_index
                for fold_index, score
                in enumerate(
                    candidate_scores
                )
                if np.isclose(
                    score,
                    minimum,
                )
            ]

            chosen_fold = int(
                rng.choice(tied)
            )

            assignment[
                well_index
            ] = chosen_fold

            fold_class_counts[
                chosen_fold
            ] += matrix[well_index]

            fold_row_counts[
                chosen_fold
            ] += matrix[
                well_index
            ].sum()

        final_score = split_score(
            fold_class_counts,
            fold_row_counts,
            class_weights,
        )

        if final_score < best_score:
            best_score = final_score
            best_assignment = (
                assignment.copy()
            )

    return (
        dict(
            zip(
                wells,
                best_assignment + 1,
            )
        ),
        float(best_score),
    )

In [ ]:
def make_one_hot_encoder():
    
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float32,
        )


def make_random_forest_pipeline(
    numeric_features,
    categorical_features,
    random_seed,
):
    transformers = []

    if numeric_features:
        numeric_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                ),
            ),
        ])

        transformers.append((
            "numeric",
            numeric_pipeline,
            numeric_features,
        ))

    if categorical_features:
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent",
                ),
            ),
            (
                "onehot",
                make_one_hot_encoder(),
            ),
        ])

        transformers.append((
            "categorical",
            categorical_pipeline,
            categorical_features,
        ))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
        verbose_feature_names_out=True,
    )

    classifier = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=(
            RF_MIN_SAMPLES_SPLIT
        ),
        min_samples_leaf=(
            RF_MIN_SAMPLES_LEAF
        ),
        max_features=RF_MAX_FEATURES,
        bootstrap=RF_BOOTSTRAP,
        n_jobs=RF_N_JOBS,
        random_state=int(
            random_seed
        ),
        class_weight=None,
        verbose=1,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])


def make_catboost_model(
    iterations,
    random_seed,
):
    return CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=int(iterations),
        learning_rate=(
            CATBOOST_LEARNING_RATE
        ),
        depth=CATBOOST_DEPTH,
        l2_leaf_reg=(
            CATBOOST_L2_LEAF_REG
        ),
        random_strength=(
            CATBOOST_RANDOM_STRENGTH
        ),
        border_count=(
            CATBOOST_BORDER_COUNT
        ),
        auto_class_weights=(
            "SqrtBalanced"
        ),
        random_seed=int(
            random_seed
        ),
        thread_count=(
            CATBOOST_THREADS
        ),
        allow_writing_files=False,
        verbose=(
            CATBOOST_VERBOSE_EVERY
        ),
    )

In [ ]:
def align_probabilities(
    raw_probabilities,
    trained_classes,
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(CLASS_CODES),
        ),
        dtype=np.float32,
    )

    for source_column, class_index in enumerate(
        trained_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = raw_probabilities[
            :,
            source_column,
        ]

    row_sums = aligned.sum(
        axis=1,
        keepdims=True,
    )

    aligned = np.divide(
        aligned,
        row_sums,
        out=np.zeros_like(aligned),
        where=row_sums > 0,
    )

    return aligned

In [ ]:
def multiclass_brier_score(
    y_true,
    probabilities,
):
    one_hot = np.eye(
        len(CLASS_CODES),
        dtype=np.float32,
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot
                ) ** 2,
                axis=1,
            )
        )
    )


def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=15,
):
    confidence = probabilities.max(
        axis=1
    )

    predictions = probabilities.argmax(
        axis=1
    )

    correctness = (
        predictions == y_true
    ).astype(float)

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for bin_index in range(n_bins):
        lower = edges[bin_index]
        upper = edges[
            bin_index + 1
        ]

        if bin_index == n_bins - 1:
            mask = (
                (confidence >= lower)
                & (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if not np.any(mask):
            continue

        ece += float(mask.mean()) * abs(
            float(
                correctness[mask].mean()
            )
            - float(
                confidence[mask].mean()
            )
        )

    return float(ece)


def calculate_global_metrics(
    y_true,
    y_pred,
    probabilities,
):
    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(len(CLASS_CODES))
                ),
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(len(CLASS_CODES))
                ),
                average="weighted",
                zero_division=0,
            )
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                labels=list(
                    range(len(CLASS_CODES))
                ),
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                labels=list(
                    range(len(CLASS_CODES))
                ),
                average="macro",
                zero_division=0,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=list(
                    range(len(CLASS_CODES))
                ),
            )
        ),
        "multiclass_brier": (
            multiclass_brier_score(
                y_true,
                probabilities,
            )
        ),
        "ECE_15_bins": (
            expected_calibration_error(
                y_true,
                probabilities,
                n_bins=15,
            )
        ),
    }


def calculate_per_class_metrics(
    y_true,
    y_pred,
):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(len(CLASS_CODES))
            ),
            zero_division=0,
        )
    )

    return pd.DataFrame({
        "class_index": list(
            range(len(CLASS_CODES))
        ),
        "lithology_code": CLASS_CODES,
        "lithology_name": CLASS_NAMES,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })


def calculate_per_well_metrics(
    evaluation_meta,
    probabilities,
):
    rows = []

    for well_name, indices in (
        evaluation_meta
        .groupby("WELL")
        .groups
        .items()
    ):
        positions = np.asarray(
            list(indices),
            dtype=int,
        )

        y_true = (
            evaluation_meta.loc[
                positions,
                "target_index",
            ]
            .to_numpy(dtype=int)
        )

        well_probabilities = (
            probabilities[positions]
        )

        y_pred = (
            well_probabilities.argmax(
                axis=1
            )
        )

        present_labels = sorted(
            np.unique(y_true).tolist()
        )

        rows.append({
            "WELL": well_name,
            "row_count": int(
                len(positions)
            ),
            "class_count": int(
                len(present_labels)
            ),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    y_pred,
                )
            ),
            "macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "log_loss": float(
                log_loss(
                    y_true,
                    well_probabilities,
                    labels=list(
                        range(len(CLASS_CODES))
                    ),
                )
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
all_fold_metrics = []
all_well_metrics = []
all_class_metrics = []
all_importances = []
all_resampling_reports = []
run_manifest = []

for model_family, feature_set_name in ACTIVE_RUNS:
    print("\n" + "=" * 110)
    print(
        "MODEL:",
        model_family,
        "| FEATURE SET:",
        feature_set_name,
    )
    print("=" * 110)

    (
        feature_frame,
        numeric_features,
        categorical_features,
    ) = prepare_feature_set(
        modeling_data,
        feature_set_name,
    )

    feature_names = (
        feature_frame.columns.tolist()
    )

    categorical_indices = [
        feature_names.index(name)
        for name in categorical_features
    ]

    for outer_fold in ACTIVE_OUTER_FOLDS:
        run_name = (
            f"{model_family}"
            f"__{feature_set_name}"
            f"__outer_fold_{outer_fold}"
        )

        print("\n" + "-" * 100)
        print("RUN:", run_name)
        print("-" * 100)

        model_extension = (
            ".joblib"
            if model_family
            == "RandomForest"
            else ".cbm"
        )

        model_path = (
            MODEL_DIR
            / f"{run_name}{model_extension}"
        )

        metric_path = (
            METRIC_DIR
            / f"{run_name}.json"
        )

        well_metric_path = (
            METRIC_DIR
            / f"{run_name}__well.csv"
        )

        class_metric_path = (
            METRIC_DIR
            / f"{run_name}__class.csv"
        )

        importance_path = (
            IMPORTANCE_DIR
            / f"{run_name}.csv"
        )

        prediction_path = (
            PREDICTION_DIR
            / f"{run_name}.parquet"
        )

        if (
            SKIP_COMPLETED_RUNS
            and model_path.exists()
            and metric_path.exists()
            and well_metric_path.exists()
            and class_metric_path.exists()
            and importance_path.exists()
            and (
                prediction_path.exists()
                or not SAVE_OOF_PROBABILITIES
            )
        ):
            print(
                "Completed artifacts found. Skipping."
            )

            with open(
                metric_path,
                "r",
                encoding="utf-8",
            ) as file:
                saved_metrics = json.load(
                    file
                )

            all_fold_metrics.append(
                saved_metrics
            )

            saved_well = pd.read_csv(
                well_metric_path
            )

            saved_well[
                "model_family"
            ] = model_family

            saved_well[
                "feature_set"
            ] = feature_set_name

            saved_well[
                "outer_fold"
            ] = outer_fold

            all_well_metrics.append(
                saved_well
            )

            saved_class = pd.read_csv(
                class_metric_path
            )

            saved_class[
                "model_family"
            ] = model_family

            saved_class[
                "feature_set"
            ] = feature_set_name

            saved_class[
                "outer_fold"
            ] = outer_fold

            all_class_metrics.append(
                saved_class
            )

            continue

        outer_valid_mask = (
            modeling_data[
                "GROUP_FOLD"
            ].to_numpy()
            == outer_fold
        )

        outer_train_mask = (
            ~outer_valid_mask
        )

        outer_train_positions = np.flatnonzero(
            outer_train_mask
        )

        outer_valid_positions = np.flatnonzero(
            outer_valid_mask
        )

        outer_train_meta = (
            modeling_data.iloc[
                outer_train_positions
            ]
            .copy()
            .reset_index(drop=True)
        )

        outer_valid_meta = (
            modeling_data.iloc[
                outer_valid_positions
            ]
            .copy()
            .reset_index(drop=True)
        )

        train_wells = set(
            outer_train_meta["WELL"]
        )

        valid_wells = set(
            outer_valid_meta["WELL"]
        )

        if train_wells & valid_wells:
            raise RuntimeError(
                "Outer WELL leakage detected."
            )

        y_outer_valid = (
            outer_valid_meta[
                "target_index"
            ]
            .to_numpy(dtype=int)
        )

        seed = (
            RANDOM_STATE
            + 1000 * outer_fold
            + sum(
                ord(character)
                for character
                in (
                    model_family
                    + feature_set_name
                )
            )
        )

        selected_iterations = None
        inner_split_score = None
        selection_seconds = 0.0

        if model_family == "RandomForest":
            (
                resampled_local_positions,
                resampling_report,
            ) = (
                apply_prior_paper_hybrid_resampling(
                    outer_train_meta,
                    random_state=seed,
                )
            )

            resampled_global_positions = (
                outer_train_positions[
                    resampled_local_positions
                ]
            )

            X_train = feature_frame.iloc[
                resampled_global_positions
            ]

            y_train = (
                modeling_data.iloc[
                    resampled_global_positions
                ]["target_index"]
                .to_numpy(dtype=int)
            )

            X_valid = feature_frame.iloc[
                outer_valid_positions
            ]

            model = (
                make_random_forest_pipeline(
                    numeric_features=(
                        numeric_features
                    ),
                    categorical_features=(
                        categorical_features
                    ),
                    random_seed=seed,
                )
            )

            fit_start = time.time()

            model.fit(
                X_train,
                y_train,
            )

            final_fit_seconds = (
                time.time()
                - fit_start
            )

            raw_probabilities = (
                model.predict_proba(
                    X_valid
                )
            )

            trained_classes = (
                model.named_steps[
                    "classifier"
                ].classes_
            )

            probabilities = (
                align_probabilities(
                    raw_probabilities,
                    trained_classes,
                )
            )

            joblib.dump(
                model,
                model_path,
                compress=3,
            )

            transformed_names = (
                model.named_steps[
                    "preprocessor"
                ].get_feature_names_out()
            )

            importance_values = (
                model.named_steps[
                    "classifier"
                ].feature_importances_
            )

            feature_importance = (
                pd.DataFrame({
                    "transformed_feature": (
                        transformed_names
                    ),
                    "importance": (
                        importance_values
                    ),
                })
                .sort_values(
                    "importance",
                    ascending=False,
                )
                .reset_index(drop=True)
            )

            resampling_report[
                "model_family"
            ] = model_family

            resampling_report[
                "feature_set"
            ] = feature_set_name

            resampling_report[
                "outer_fold"
            ] = outer_fold

            resampling_report.to_csv(
                RESAMPLING_DIR
                / f"{run_name}.csv",
                index=False,
            )

            all_resampling_reports.append(
                resampling_report
            )

            resampled_train_rows = int(
                len(
                    resampled_global_positions
                )
            )

            del (
                X_train,
                X_valid,
                raw_probabilities,
                model,
            )

        elif model_family == "CatBoost":
            (
                inner_fold_map,
                inner_split_score,
            ) = build_grouped_inner_folds(
                outer_train_meta,
                n_folds=3,
                random_state=seed,
                n_restarts=100,
            )

            outer_train_meta[
                "_INNER_FOLD"
            ] = (
                outer_train_meta[
                    "WELL"
                ]
                .map(inner_fold_map)
                .astype("int8")
            )

            preferred_inner_fold = (
                ((outer_fold - 1) % 3)
                + 1
            )

            available_inner_folds = sorted(
                outer_train_meta[
                    "_INNER_FOLD"
                ]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )

            candidate_inner_folds = (
                [preferred_inner_fold]
                + [
                    fold_id
                    for fold_id in available_inner_folds
                    if fold_id != preferred_inner_fold
                ]
            )

            inner_candidate_report = []
            inner_validation_fold = None

            for candidate_fold in candidate_inner_folds:
                candidate_valid_mask = (
                    outer_train_meta[
                        "_INNER_FOLD"
                    ].to_numpy()
                    == candidate_fold
                )

                candidate_train_mask = (
                    ~candidate_valid_mask
                )

                train_classes_candidate = set(
                    outer_train_meta.loc[
                        candidate_train_mask,
                        "target_index",
                    ]
                    .astype(int)
                    .unique()
                    .tolist()
                )

                valid_classes_candidate = set(
                    outer_train_meta.loc[
                        candidate_valid_mask,
                        "target_index",
                    ]
                    .astype(int)
                    .unique()
                    .tolist()
                )

                missing_from_train = sorted(
                    valid_classes_candidate
                    - train_classes_candidate
                )

                inner_candidate_report.append({
                    "candidate_inner_fold": int(
                        candidate_fold
                    ),
                    "train_rows": int(
                        candidate_train_mask.sum()
                    ),
                    "validation_rows": int(
                        candidate_valid_mask.sum()
                    ),
                    "train_wells": int(
                        outer_train_meta.loc[
                            candidate_train_mask,
                            "WELL",
                        ].nunique()
                    ),
                    "validation_wells": int(
                        outer_train_meta.loc[
                            candidate_valid_mask,
                            "WELL",
                        ].nunique()
                    ),
                    "missing_class_indices_in_train": (
                        missing_from_train
                    ),
                    "is_class_safe": (
                        len(missing_from_train) == 0
                    ),
                })

                if (
                    candidate_train_mask.any()
                    and candidate_valid_mask.any()
                    and len(missing_from_train) == 0
                ):
                    inner_validation_fold = int(
                        candidate_fold
                    )
                    break

            inner_candidate_report_df = pd.DataFrame(
                inner_candidate_report
            )

            print(
                "Inner validation candidate audit:"
            )
            display(
                inner_candidate_report_df
            )

            if inner_validation_fold is None:
                raise RuntimeError(
                    "No class-safe inner validation fold was found. "
                    "Every candidate validation fold contains at least "
                    "one class absent from its corresponding training set."
                )

            print(
                "Selected class-safe inner validation fold:",
                inner_validation_fold,
            )

            inner_valid_local_mask = (
                outer_train_meta[
                    "_INNER_FOLD"
                ].to_numpy()
                == inner_validation_fold
            )

            inner_train_local_mask = (
                ~inner_valid_local_mask
            )

            inner_train_global = (
                outer_train_positions[
                    inner_train_local_mask
                ]
            )

            inner_valid_global = (
                outer_train_positions[
                    inner_valid_local_mask
                ]
            )

            inner_train_wells = set(
                outer_train_meta.loc[
                    inner_train_local_mask,
                    "WELL",
                ]
            )

            inner_valid_wells = set(
                outer_train_meta.loc[
                    inner_valid_local_mask,
                    "WELL",
                ]
            )

            if (
                inner_train_wells
                & inner_valid_wells
            ):
                raise RuntimeError(
                    "Inner WELL leakage detected."
                )

            final_inner_train_classes = set(
                outer_train_meta.loc[
                    inner_train_local_mask,
                    "target_index",
                ]
                .astype(int)
                .unique()
                .tolist()
            )

            final_inner_valid_classes = set(
                outer_train_meta.loc[
                    inner_valid_local_mask,
                    "target_index",
                ]
                .astype(int)
                .unique()
                .tolist()
            )

            unseen_inner_validation_classes = sorted(
                final_inner_valid_classes
                - final_inner_train_classes
            )

            if unseen_inner_validation_classes:
                raise RuntimeError(
                    "Class-safe selection failed. "
                    "Validation classes absent from training: "
                    f"{unseen_inner_validation_classes}"
                )

            selection_train_pool = Pool(
                data=feature_frame.iloc[
                    inner_train_global
                ],
                label=(
                    modeling_data.iloc[
                        inner_train_global
                    ]["target_index"]
                    .to_numpy(dtype=int)
                ),
                cat_features=(
                    categorical_indices
                ),
                feature_names=feature_names,
            )

            selection_valid_pool = Pool(
                data=feature_frame.iloc[
                    inner_valid_global
                ],
                label=(
                    modeling_data.iloc[
                        inner_valid_global
                    ]["target_index"]
                    .to_numpy(dtype=int)
                ),
                cat_features=(
                    categorical_indices
                ),
                feature_names=feature_names,
            )

            selection_model = (
                make_catboost_model(
                    iterations=(
                        CATBOOST_MAX_ITERATIONS
                    ),
                    random_seed=seed,
                )
            )

            selection_start = time.time()

            selection_model.fit(
                selection_train_pool,
                eval_set=(
                    selection_valid_pool
                ),
                use_best_model=True,
                early_stopping_rounds=(
                    CATBOOST_EARLY_STOPPING
                ),
            )

            selection_seconds = (
                time.time()
                - selection_start
            )

            best_iteration = (
                selection_model.get_best_iteration()
            )

            if (
                best_iteration is None
                or best_iteration < 0
            ):
                best_iteration = (
                    CATBOOST_MAX_ITERATIONS
                    - 1
                )

            selected_iterations = int(
                np.clip(
                    best_iteration + 1,
                    MIN_CATBOOST_REFIT_ITERATIONS,
                    CATBOOST_MAX_ITERATIONS,
                )
            )

            del (
                selection_train_pool,
                selection_valid_pool,
                selection_model,
            )

            gc.collect()

            outer_train_pool = Pool(
                data=feature_frame.iloc[
                    outer_train_positions
                ],
                label=(
                    outer_train_meta[
                        "target_index"
                    ]
                    .to_numpy(dtype=int)
                ),
                cat_features=(
                    categorical_indices
                ),
                feature_names=feature_names,
            )

            outer_valid_pool = Pool(
                data=feature_frame.iloc[
                    outer_valid_positions
                ],
                cat_features=(
                    categorical_indices
                ),
                feature_names=feature_names,
            )

            model = make_catboost_model(
                iterations=(
                    selected_iterations
                ),
                random_seed=seed + 17,
            )

            fit_start = time.time()

            model.fit(
                outer_train_pool
            )

            final_fit_seconds = (
                time.time()
                - fit_start
            )

            raw_probabilities = (
                model.predict_proba(
                    outer_valid_pool
                )
            )

            probabilities = (
                align_probabilities(
                    raw_probabilities,
                    model.classes_,
                )
            )

            model.save_model(
                str(model_path)
            )

            feature_importance = (
                pd.DataFrame({
                    "transformed_feature": (
                        feature_names
                    ),
                    "importance": (
                        model.get_feature_importance(
                            outer_train_pool
                        )
                    ),
                })
                .sort_values(
                    "importance",
                    ascending=False,
                )
                .reset_index(drop=True)
            )

            resampled_train_rows = int(
                len(outer_train_positions)
            )

            del (
                outer_train_pool,
                outer_valid_pool,
                raw_probabilities,
                model,
            )

        else:
            raise ValueError(
                f"Unknown model family: {model_family}"
            )

        predicted_index = (
            probabilities.argmax(axis=1)
            .astype(int)
        )

        fold_metrics = (
            calculate_global_metrics(
                y_outer_valid,
                predicted_index,
                probabilities,
            )
        )

        evaluation_meta = (
            outer_valid_meta[
                [
                    "ROW_ID",
                    "WELL",
                    "DEPTH_MD",
                    "target",
                    "target_index",
                    "GROUP_FOLD",
                    "GEO_BLOCK",
                ]
            ]
            .copy()
            .reset_index(drop=True)
        )

        per_well = (
            calculate_per_well_metrics(
                evaluation_meta,
                probabilities,
            )
        )

        per_class = (
            calculate_per_class_metrics(
                y_outer_valid,
                predicted_index,
            )
        )

        fold_metrics.update({
            "run_mode": RUN_MODE,
            "model_family": model_family,
            "feature_set": feature_set_name,
            "scientific_role": (
                FEATURE_SET_DEFINITIONS[
                    feature_set_name
                ]["scientific_role"]
            ),
            "outer_fold": int(
                outer_fold
            ),
            "outer_train_rows_before_resampling": int(
                len(outer_train_positions)
            ),
            "model_training_rows": int(
                resampled_train_rows
            ),
            "outer_validation_rows": int(
                len(outer_valid_positions)
            ),
            "train_wells": int(
                len(train_wells)
            ),
            "validation_wells": int(
                len(valid_wells)
            ),
            "raw_feature_count": int(
                len(feature_names)
            ),
            "selected_catboost_iterations": (
                int(selected_iterations)
                if selected_iterations
                is not None
                else None
            ),
            "inner_split_score": (
                float(inner_split_score)
                if inner_split_score
                is not None
                else None
            ),
            "selection_fit_seconds": float(
                selection_seconds
            ),
            "final_fit_seconds": float(
                final_fit_seconds
            ),
            "mean_well_macro_f1": float(
                per_well[
                    "macro_f1_present_classes"
                ].mean()
            ),
            "median_well_macro_f1": float(
                per_well[
                    "macro_f1_present_classes"
                ].median()
            ),
            "worst_well_macro_f1": float(
                per_well[
                    "macro_f1_present_classes"
                ].min()
            ),
            "mean_well_log_loss": float(
                per_well[
                    "log_loss"
                ].mean()
            ),
        })

        with open(
            metric_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                fold_metrics,
                file,
                ensure_ascii=False,
                indent=2,
            )

        per_well.to_csv(
            well_metric_path,
            index=False,
        )

        per_class.to_csv(
            class_metric_path,
            index=False,
        )

        feature_importance.to_csv(
            importance_path,
            index=False,
        )

        if SAVE_OOF_PROBABILITIES:
            prediction_frame = (
                evaluation_meta.copy()
            )

            prediction_frame[
                "predicted_index"
            ] = predicted_index

            prediction_frame[
                "predicted_code"
            ] = [
                INDEX_TO_CLASS[
                    int(index)
                ]
                for index in predicted_index
            ]

            for class_index, class_code in enumerate(
                CLASS_CODES
            ):
                prediction_frame[
                    f"prob_{class_code}"
                ] = probabilities[
                    :,
                    class_index,
                ]

            prediction_frame.to_parquet(
                prediction_path,
                index=False,
            )

        all_fold_metrics.append(
            fold_metrics
        )

        per_well[
            "model_family"
        ] = model_family

        per_well[
            "feature_set"
        ] = feature_set_name

        per_well[
            "outer_fold"
        ] = outer_fold

        all_well_metrics.append(
            per_well
        )

        per_class[
            "model_family"
        ] = model_family

        per_class[
            "feature_set"
        ] = feature_set_name

        per_class[
            "outer_fold"
        ] = outer_fold

        all_class_metrics.append(
            per_class
        )

        feature_importance[
            "model_family"
        ] = model_family

        feature_importance[
            "feature_set"
        ] = feature_set_name

        feature_importance[
            "outer_fold"
        ] = outer_fold

        all_importances.append(
            feature_importance
        )

        run_manifest.append({
            "run_name": run_name,
            "model_family": model_family,
            "feature_set": feature_set_name,
            "outer_fold": int(
                outer_fold
            ),
            "model_path": str(
                model_path
            ),
            "prediction_path": (
                str(prediction_path)
                if SAVE_OOF_PROBABILITIES
                else None
            ),
            "status": "completed",
        })

        print(
            json.dumps(
                fold_metrics,
                indent=2,
            )
        )

        del (
            probabilities,
            predicted_index,
            feature_importance,
            per_well,
            per_class,
        )

        gc.collect()

    del feature_frame
    gc.collect()

In [ ]:
fold_metrics_df = pd.DataFrame(
    all_fold_metrics
)

well_metrics_df = (
    pd.concat(
        all_well_metrics,
        ignore_index=True,
    )
    if all_well_metrics
    else pd.DataFrame()
)

class_metrics_df = (
    pd.concat(
        all_class_metrics,
        ignore_index=True,
    )
    if all_class_metrics
    else pd.DataFrame()
)

importance_df = (
    pd.concat(
        all_importances,
        ignore_index=True,
    )
    if all_importances
    else pd.DataFrame()
)

resampling_reports_df = (
    pd.concat(
        all_resampling_reports,
        ignore_index=True,
    )
    if all_resampling_reports
    else pd.DataFrame()
)

display(fold_metrics_df)

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall",
    "log_loss",
    "multiclass_brier",
    "ECE_15_bins",
    "mean_well_macro_f1",
    "median_well_macro_f1",
    "worst_well_macro_f1",
    "mean_well_log_loss",
]

available_metrics = [
    column
    for column in METRIC_COLUMNS
    if column in fold_metrics_df.columns
]

model_summary = (
    fold_metrics_df
    .groupby(
        [
            "run_mode",
            "model_family",
            "feature_set",
            "scientific_role",
        ]
    )[available_metrics]
    .agg(
        [
            "mean",
            "std",
            "min",
            "max",
        ]
    )
    .reset_index()
)

model_summary.columns = [
    (
        "_".join(
            str(part)
            for part in column
            if str(part)
        )
        if isinstance(column, tuple)
        else column
    )
    for column in model_summary.columns
]

display(model_summary)

In [ ]:
if not importance_df.empty:
    mean_importance = (
        importance_df
        .groupby(
            [
                "model_family",
                "feature_set",
                "transformed_feature",
            ],
            as_index=False,
        )
        .agg(
            mean_importance=(
                "importance",
                "mean",
            ),
            std_importance=(
                "importance",
                "std",
            ),
            fold_count=(
                "outer_fold",
                "nunique",
            ),
        )
        .sort_values(
            [
                "model_family",
                "feature_set",
                "mean_importance",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
    )

    display(
        mean_importance.groupby(
            [
                "model_family",
                "feature_set",
            ]
        ).head(20)
    )
else:
    mean_importance = pd.DataFrame()

In [ ]:
for model_family, feature_set_name in ACTIVE_RUNS:
    prediction_frames = []

    for outer_fold in ACTIVE_OUTER_FOLDS:
        path = (
            PREDICTION_DIR
            / (
                f"{model_family}"
                f"__{feature_set_name}"
                f"__outer_fold_{outer_fold}"
                ".parquet"
            )
        )

        if path.exists():
            prediction_frames.append(
                pd.read_parquet(
                    path,
                    columns=[
                        "target_index",
                        "predicted_index",
                    ],
                )
            )

    if not prediction_frames:
        continue

    oof = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    matrix = confusion_matrix(
        oof["target_index"],
        oof["predicted_index"],
        labels=list(
            range(len(CLASS_CODES))
        ),
    )

    matrix_df = pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )

    matrix_df.to_csv(
        OUTPUT_DIR
        / (
            f"confusion_matrix"
            f"__{model_family}"
            f"__{feature_set_name}.csv"
        )
    )

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    normalized = np.divide(
        matrix,
        row_sums,
        out=np.zeros_like(
            matrix,
            dtype=float,
        ),
        where=row_sums > 0,
    )

    plt.figure(figsize=(12, 10))

    image = plt.imshow(
        normalized,
        aspect="auto",
    )

    plt.colorbar(image)

    plt.xticks(
        range(len(CLASS_NAMES)),
        CLASS_NAMES,
        rotation=90,
    )

    plt.yticks(
        range(len(CLASS_NAMES)),
        CLASS_NAMES,
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(
        f"OOF confusion matrix\n"
        f"{model_family} | {feature_set_name}"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / (
            f"confusion_matrix"
            f"__{model_family}"
            f"__{feature_set_name}.png"
        ),
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()

# 17. ذخیره خروجی نهایی

In [ ]:
class_mapping.to_csv(
    OUTPUT_DIR
    / "00_class_mapping.csv",
    index=False,
)

feature_set_table.to_csv(
    OUTPUT_DIR
    / "01_feature_set_definitions.csv",
    index=False,
)

fold_metrics_df.to_csv(
    OUTPUT_DIR
    / "02_fold_metrics.csv",
    index=False,
)

model_summary.to_csv(
    OUTPUT_DIR
    / "03_model_feature_set_summary.csv",
    index=False,
)

if not well_metrics_df.empty:
    well_metrics_df.to_csv(
        OUTPUT_DIR
        / "04_per_well_metrics.csv",
        index=False,
    )

if not class_metrics_df.empty:
    class_metrics_df.to_csv(
        OUTPUT_DIR
        / "05_per_class_metrics.csv",
        index=False,
    )

if not importance_df.empty:
    importance_df.to_csv(
        OUTPUT_DIR
        / "06_feature_importance_by_fold.csv",
        index=False,
    )

if not mean_importance.empty:
    mean_importance.to_csv(
        OUTPUT_DIR
        / "07_mean_feature_importance.csv",
        index=False,
    )

if not resampling_reports_df.empty:
    resampling_reports_df.to_csv(
        OUTPUT_DIR
        / "08_hybrid_resampling_summary.csv",
        index=False,
    )

pd.DataFrame(
    run_manifest
).to_csv(
    OUTPUT_DIR
    / "09_run_manifest.csv",
    index=False,
)

run_configuration = {
    "run_mode": RUN_MODE,
    "contract_version": (
        contract["contract_version"]
    ),
    "test_used": False,
    "outer_group": "WELL",
    "outer_folds": ACTIVE_OUTER_FOLDS,
    "active_runs": [
        {
            "model_family": model_family,
            "feature_set": feature_set,
        }
        for model_family, feature_set
        in ACTIVE_RUNS
    ],
    "random_forest_primary": {
        "n_estimators": RF_N_ESTIMATORS,
        "full_n_estimators": (
            FULL_RF_N_ESTIMATORS
        ),
        "max_depth": RF_MAX_DEPTH,
        "min_samples_split": (
            RF_MIN_SAMPLES_SPLIT
        ),
        "min_samples_leaf": (
            RF_MIN_SAMPLES_LEAF
        ),
        "max_features": RF_MAX_FEATURES,
        "n_jobs": RF_N_JOBS,
        "hybrid_resampling": {
            "undersample_fractions": (
                UNDERSAMPLE_FRACTIONS
            ),
            "oversample_multipliers": (
                OVERSAMPLE_MULTIPLIERS
            ),
            "training_only": True,
        },
    },
    "catboost_comparator": {
        "max_iterations": (
            CATBOOST_MAX_ITERATIONS
        ),
        "depth": CATBOOST_DEPTH,
        "learning_rate": (
            CATBOOST_LEARNING_RATE
        ),
        "auto_class_weights": (
            "SqrtBalanced"
        ),
        "inner_grouped_validation": True,
    },
}

with open(
    CONFIG_DIR
    / "run_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_configuration,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("All outputs saved in:")
print(OUTPUT_DIR)